# UCF-Crime CLIP Feature Extractor (Colab)

Iki mod:
- **`single_zip`** (ONERILEN, tam otomatik): resmi tek zip'i indirir, ICINDEN videolari
  tek tek akitarak (stream) isler -> diski sismez. Indirme kopsa OTOMATIK devam eder.
- **`parts`**: parca parca indir/isle/sil.

Extractor `src/clip_features.py` ile **birebir ayni**: open_clip ViT-B/16, frame basina
L2-normalize.

**Ablation**: `FPS_LIST = [8, 4, 2, 1]` -> TEK run'da hepsi cikarilir, her fps OTOMATIK ayri
klasore yazilir (`features_fps8`, `features_fps4`, ...). Zip bir kez acilir, videolar her fps
icin tekrar islenir. Resume var (her fps kendi klasorunde kaldigi yerden devam eder).

Cikti:
```
features_fps8/{train,test}/<ClassName>/<video>.npy   # (T, 512), normalize
features_fps8/frame_counts.json                       # stem -> {frames, fps}
features_fps4/...  features_fps2/...  features_fps1/...
meta/                                                 # zip icinden cikan .txt (fps'ten bagimsiz)
```

## 1) Kurulum (Runtime > GPU: T4/L4)

In [ ]:
!pip -q install open_clip_torch==2.24.0 opencv-python-headless tqdm gdown
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) CONFIG

In [ ]:
# ===== CONFIG =====
MODE = 'single_zip'        # 'single_zip'  veya  'parts'

# --- ABLATION: tek run'da hangi fps'ler cikarilsin ---
FPS_LIST = [8, 4, 2, 1]    # her fps OTOMATIK ayri klasore yazilir (asagidaki sablona gore)

# Klasor ismi otomatik: {fps} yerine sayi gelir -> features_fps8 / _fps4 / _fps2 / _fps1
OUT_ROOT_TMPL = '/content/drive/MyDrive/UCF_Crime/features_fps{fps}'   # CIKTI -> Drive
META_DIR      = '/content/drive/MyDrive/UCF_Crime/meta'                # fps'ten bagimsiz, ortak
WORK_DIR      = '/content/work'                                        # gecici yerel alan

# (run hucresi bunlari her fps'te otomatik gunceller; ilk degerler sadece varsayilan)
SAMPLE_FPS = FPS_LIST[0]
OUT_ROOT   = OUT_ROOT_TMPL.format(fps=SAMPLE_FPS)

# --- single_zip ayarlari ---
SINGLE_ZIP_URL = 'https://www.crcv.ucf.edu/data1/chenchen/UCF_Crimes.zip'
LOCAL_ZIP      = '/content/UCF_Crimes.zip'
EXPECTED_BYTES = 102957372377   # spider testinden gelen tam boyut (~96GB). Indirme tamamlik kontrolu.

# --- parts ayarlari (MODE='parts' ise) ---
TRAIN_LIST = '/content/drive/MyDrive/UCF_Crime/Anomaly_Train.txt'   # yoksa None
TEST_LIST  = '/content/drive/MyDrive/UCF_Crime/Anomaly_Test.txt'
PARTS = [
    # {'name':'Anomaly-1','type':'url','src':'https://www.dropbox.com/sh/.../Anomaly-Videos-Part-1.zip?dl=1'},
]

# clip_features.py ile AYNI!
CLIP_MODEL    = 'ViT-B-16'
CLIP_PRETRAIN = 'laion2b_s34b_b88k'
BATCH_SIZE    = 256
USE_FP16      = True
VIDEO_EXTS    = ('.mp4', '.avi', '.mkv', '.mov')

## 3) Model + extractor (clip_features.py ile birebir)

In [ ]:
import os, cv2, numpy as np, torch, open_clip
from pathlib import Path
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, _, preprocess = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=CLIP_PRETRAIN)
model = model.to(device).eval()
print('Model:', CLIP_MODEL, CLIP_PRETRAIN, '| feature-dim:', model.visual.output_dim)

def get_meta(path):
    cap = cv2.VideoCapture(str(path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    cap.release(); return n, fps

def extract_video_features(path, sample_fps, batch_size=BATCH_SIZE):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened(): raise RuntimeError(f'Video acilamadi: {path}')
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    if fps <= 0: fps = 25
    step = max(int(round(fps / sample_fps)), 1)
    feats, buf, idx = [], [], 0
    def flush():
        if not buf: return
        batch = torch.stack(buf).to(device)
        with torch.no_grad():
            if USE_FP16 and device == 'cuda':
                with torch.autocast('cuda', dtype=torch.float16):
                    f = model.encode_image(batch)
            else:
                f = model.encode_image(batch)
            f = f.float(); f = f / f.norm(dim=-1, keepdim=True)
        feats.append(f.cpu().numpy().astype(np.float32))
    while True:
        ret, frame = cap.read()
        if not ret: break
        if idx % step == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            buf.append(preprocess(Image.fromarray(rgb)))
            if len(buf) >= batch_size: flush(); buf.clear()
        idx += 1
    flush(); cap.release()
    if not feats: return None
    return np.concatenate(feats, axis=0)

## 4) Yardimcilar (indirme-retry, split/class, frame_counts)

In [ ]:
import subprocess, shutil, zipfile, json, glob, time

def robust_download(url, dest, expected_bytes=0, wait=15, max_hours=10):
    """Kopsa otomatik devam eder; dosya expected_bytes'a ulasana kadar dener."""
    deadline = time.time() + max_hours * 3600
    def size(): return os.path.getsize(dest) if os.path.exists(dest) else 0
    while True:
        if expected_bytes and size() >= expected_bytes:
            print(f'Indirme tamam: {size()/1e9:.1f} GB'); return
        print(f'Indirme/devam... ({size()/1e9:.1f} GB)')
        subprocess.run(['wget','-c','--no-check-certificate','--timeout=60',
                        '--read-timeout=60','--tries=20','--retry-connrefused',
                        '-O',dest,url])
        if expected_bytes and size() >= expected_bytes:
            print(f'Indirme tamam: {size()/1e9:.1f} GB'); return
        if not expected_bytes:  # boyut bilinmiyorsa tek denemeyle yetin
            return
        if time.time() > deadline:
            raise RuntimeError('Indirme zaman asimi (max_hours).')
        print(f'  koptu, {wait}s sonra devam...'); time.sleep(wait)

def parse_list_stems(list_path):
    stems = set()
    if list_path and os.path.exists(list_path):
        with open(list_path) as f:
            for line in f:
                line = line.strip().replace('\\', '/')
                if line: stems.add(Path(line.split('/')[-1]).stem)
    return stems

train_stems, test_stems = set(), set()

def classify(stem, parent):
    p = parent.lower()
    if train_stems or test_stems:
        if stem in test_stems: return 'test', parent
        if stem in train_stems: return 'train', parent
    return ('test' if 'test' in p else 'train'), parent

# NOT: OUT_ROOT / FC_PATH her fps'te run hucresinde yeniden atanir.
FC_PATH = os.path.join(OUT_ROOT, 'frame_counts.json')
def load_frame_counts():
    if os.path.exists(FC_PATH):
        with open(FC_PATH) as f: return json.load(f)
    return {}
def save_frame_counts(fc):
    os.makedirs(OUT_ROOT, exist_ok=True)
    tmp = FC_PATH + '.tmp'
    with open(tmp, 'w') as f: json.dump(fc, f)
    os.replace(tmp, FC_PATH)

def save_one(stem, parent, video_path, fc):
    split, cls = classify(stem, parent)
    out_dir = os.path.join(OUT_ROOT, split, cls); os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, stem + '.npy')
    if stem not in fc:
        try:
            n, fps = get_meta(video_path); fc[stem] = {'frames': n, 'fps': fps}
        except Exception: pass
    if os.path.exists(out_path):
        return False, 'skip'
    feats = extract_video_features(video_path, sample_fps=SAMPLE_FPS)   # global SAMPLE_FPS (run hucresi gunceller)
    if feats is None or len(feats) == 0:
        return False, 'bos'
    np.save(out_path + '.tmp.npy', feats); os.replace(out_path + '.tmp.npy', out_path)
    return True, 'ok'

def fetch_part(part, dest_dir):
    os.makedirs(dest_dir, exist_ok=True)
    t, src = part['type'], part['src']
    zip_path = os.path.join(dest_dir, part['name'] + '.zip')
    if t == 'url': robust_download(src, zip_path)
    elif t == 'gdrive': subprocess.run(['gdown','-O',zip_path,src], check=True)
    elif t == 'drive':
        if src.lower().endswith('.zip'): shutil.copy(src, zip_path)
        else: return src
    else: raise ValueError(t)
    out = os.path.join(dest_dir, 'videos'); os.makedirs(out, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z: z.extractall(out)
    os.remove(zip_path); return out

## 5) CALISTIR — FPS_LIST'teki tum fps'ler tek seferde (otomatik klasor) — calistir ve birak

In [ ]:
from tqdm import tqdm
os.makedirs(WORK_DIR, exist_ok=True)

if MODE == 'single_zip':
    print('Tek zip indiriliyor (kopsa otomatik devam)...')
    robust_download(SINGLE_ZIP_URL, LOCAL_ZIP, expected_bytes=EXPECTED_BYTES)
    z = zipfile.ZipFile(LOCAL_ZIP); names = z.namelist()

    # meta'yi bir kez cikar (fps'ten bagimsiz)
    os.makedirs(META_DIR, exist_ok=True)
    for m in names:
        if m.lower().endswith('.txt') and not m.endswith('/'):
            with z.open(m) as s: data = s.read()
            with open(os.path.join(META_DIR, Path(m).name), 'wb') as o: o.write(data)
    tr = glob.glob(os.path.join(META_DIR, '*[Tt]rain*.txt'))
    te = glob.glob(os.path.join(META_DIR, '*[Tt]est*.txt'))
    train_stems = parse_list_stems(tr[0]) if tr else set()
    test_stems  = parse_list_stems(te[0]) if te else set()
    print(f'meta txt: {len(glob.glob(os.path.join(META_DIR,"*.txt")))} | split stems train/test: {len(train_stems)}/{len(test_stems)}')

    vids = [m for m in names if m.lower().endswith(VIDEO_EXTS) and not m.endswith('/')]
    print('Toplam video:', len(vids))

    for SAMPLE_FPS in FPS_LIST:
        OUT_ROOT = OUT_ROOT_TMPL.format(fps=SAMPLE_FPS)
        FC_PATH  = os.path.join(OUT_ROOT, 'frame_counts.json')
        os.makedirs(OUT_ROOT, exist_ok=True)
        fc = load_frame_counts()
        done = skipped = 0; failed = []
        print(f'\n##### FPS={SAMPLE_FPS}  ->  {OUT_ROOT}')
        for i, m in enumerate(tqdm(vids, desc=f'fps{SAMPLE_FPS}')):
            stem = Path(m).stem; parent = Path(m).parent.name
            tmp = os.path.join(WORK_DIR, Path(m).name)
            try:
                with z.open(m) as s, open(tmp, 'wb') as o: shutil.copyfileobj(s, o)
                ok, msg = save_one(stem, parent, tmp, fc)
                if ok: done += 1
                elif msg == 'skip': skipped += 1
                else: failed.append((m, msg))
            except Exception as e:
                failed.append((m, str(e)))
            finally:
                if os.path.exists(tmp): os.remove(tmp)
            if i % 50 == 0: save_frame_counts(fc)
        save_frame_counts(fc)
        print(f'  FPS={SAMPLE_FPS} bitti -> yeni:{done} | atlanan:{skipped} | hata:{len(failed)} | frame_counts:{len(fc)}')
        for p, e in failed[:5]: print('   -', p, '->', e)

elif MODE == 'parts':
    train_stems = parse_list_stems(TRAIN_LIST); test_stems = parse_list_stems(TEST_LIST)
    print(f'split stems train/test: {len(train_stems)}/{len(test_stems)}')
    for SAMPLE_FPS in FPS_LIST:
        OUT_ROOT = OUT_ROOT_TMPL.format(fps=SAMPLE_FPS)
        FC_PATH  = os.path.join(OUT_ROOT, 'frame_counts.json')
        os.makedirs(OUT_ROOT, exist_ok=True)
        fc = load_frame_counts()
        done = skipped = 0; failed = []
        print(f'\n##### FPS={SAMPLE_FPS}  ->  {OUT_ROOT}')
        for part in PARTS:
            print('  PARCA:', part['name'])
            part_dir = os.path.join(WORK_DIR, part['name'])
            try:
                vdir = fetch_part(part, part_dir)
                files = [os.path.join(r,f) for r,_,fs in os.walk(vdir) for f in fs if f.lower().endswith(VIDEO_EXTS)]
                for vp in tqdm(files, desc=f'  fps{SAMPLE_FPS}'):
                    stem = Path(vp).stem; parent = Path(vp).parent.name
                    try:
                        ok, msg = save_one(stem, parent, vp, fc)
                        if ok: done += 1
                        elif msg == 'skip': skipped += 1
                        else: failed.append((vp, msg))
                    except Exception as e:
                        failed.append((vp, str(e)))
            except Exception as e:
                print('  PARCA HATASI:', e)
            finally:
                save_frame_counts(fc); shutil.rmtree(part_dir, ignore_errors=True)
        print(f'  FPS={SAMPLE_FPS} bitti -> yeni:{done} | atlanan:{skipped} | hata:{len(failed)}')

print('\n=== HEPSI BITTI ===  fps listesi:', FPS_LIST)

## 6) Sanity check (her fps icin)

In [ ]:
from collections import Counter
for FPS in FPS_LIST:
    root = OUT_ROOT_TMPL.format(fps=FPS)
    files = glob.glob(os.path.join(root, '*', '*', '*.npy'))
    print(f'\nfps{FPS}: {root}')
    print('  Feature dosyasi:', len(files))
    if files:
        print('  Split dagilimi:', Counter(Path(f).parts[-3] for f in files))
        arr = np.load(files[0])
        print('  Ornek:', Path(files[0]).name, '| shape:', arr.shape, '| dtype:', arr.dtype)
        print('  Frame norm ort (1.0 bekleniyor):', float(np.linalg.norm(arr, axis=1).mean()))
# os.remove(LOCAL_ZIP)   # isin bitince yerel zip'i sil

## Sonraki adim (kendi makinende) — 8/4/2/1 tablosu icin her fps ayri
Drive'dan `features_fps8/`, `features_fps4/`, `features_fps2/`, `features_fps1/` (+ her birinin
`frame_counts.json`'u) ve `meta/` -> projeye. Sonra her fps icin (ornek fps8):
```bash
python train_cliptsa_ucf.py --feature_root data/ucf/features_fps8 --epochs 10

python evaluate_cliptsa_ucf.py --feature_root data/ucf/features_fps8/test \
  --checkpoint checkpoints/ucf/cliptsa_ucf_best.pkl                       # video-level

python evaluate_frame_level_auc.py \
  --annotation data/ucf/meta/Temporal_Anomaly_Annotation_for_Testing_Videos.txt \
  --feature_root data/ucf/features_fps8/test \
  --frame_counts data/ucf/features_fps8/frame_counts.json \
  --checkpoint checkpoints/ucf/cliptsa_ucf_best.pkl                       # frame-level (SOTA ~0.87)
```